In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
from torchvision.models.segmentation import deeplabv3_resnet50
import numpy as np
import matplotlib.pyplot as plt
import os
import json
import csv
from copy import deepcopy
import cv2

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

os.makedirs("artifacts/figures", exist_ok=True)

Using device: cpu


In [12]:
# Список для хранения результатов экспериментов
runs_data = []

def log_run(exp_id, task, dataset, model_summary, optimizer, lr, epochs, 
            best_val_acc, test_acc, mean_iou=None, notes=""):
    runs_data.append({
        "experiment_id": exp_id,
        "task": task,
        "dataset": dataset,
        "seed": SEED,
        "model_summary": model_summary,
        "optimizer": optimizer,
        "lr": lr,
        "epochs_trained": epochs,
        "best_val_accuracy": round(best_val_acc, 4),
        "test_accuracy": round(test_acc, 4) if test_acc is not None else "",
        "precision": "",
        "recall": "",
        "mean_iou": round(mean_iou, 4) if mean_iou is not None else "",
        "notes": notes
    })

def save_runs():
    with open("artifacts/runs.csv", "w", newline="") as f:
        if runs_data:
            writer = csv.DictWriter(f, fieldnames=runs_data[0].keys())
            writer.writeheader()
            writer.writerows(runs_data)
    print("Saved artifacts/runs.csv")

In [13]:
# CIFAR100 Mean и Std для нормализации
CIFAR100_MEAN = [0.5071, 0.4867, 0.4408]
CIFAR100_STD = [0.2675, 0.2586, 0.2762]

# Базовый трансформ (без аугментаций)
transform_base = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD)
])

# Трансформ с аугментациями
transform_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD)
])

# Трансформ для ResNet (ImageNet статистики)
transform_resnet = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [21]:
# Загружаем полный train датасет БЕЗ трансформов
train_dataset_full = torchvision.datasets.CIFAR100(
    root="./data", train=True, download=True, transform=None  # <-- Важно: None
)
test_dataset = torchvision.datasets.CIFAR100(
    root="./data", train=False, download=True, transform=None  # <-- Важно: None
)

# Делим train на train/val (80/20)
n_train = len(train_dataset_full)
indices = list(range(n_train))
np.random.shuffle(indices)
split = int(np.floor(0.8 * n_train))
train_indices, val_indices = indices[:split], indices[split:]

# Создаём сабсеты
train_subset = Subset(train_dataset_full, train_indices)
val_subset = Subset(train_dataset_full, val_indices)

print(f"Train: {len(train_indices)}, Val: {len(val_indices)}, Test: {len(test_dataset)}")

# Sanity check
img, label = train_dataset_full[0]
print(f"Sample type: {type(img)}, Label: {label}")

Train: 40000, Val: 10000, Test: 10000
Sample type: <class 'PIL.Image.Image'>, Label: 19


In [22]:
class SubsetTransform(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    
    def __getitem__(self, idx):
        img, label = self.subset[idx]
        # img теперь PIL Image, можно применять все трансформы
        if self.transform:
            img = self.transform(img)
        return img, label
    
    def __len__(self):
        return len(self.subset)

In [16]:
# Простая CNN для экспериментов C1 и C2
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=100):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


# Функция для создания ResNet18
def get_resnet18(pretrained=True, num_classes=100, freeze_backbone=False):
    model = resnet18(
        weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    )
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
        for param in model.fc.parameters():
            param.requires_grad = True
            
    return model

In [17]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
    return total_loss / total, 100. * correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            total_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            
    return total_loss / total, 100. * correct / total

In [23]:
def run_experiment(exp_id, train_tf, val_tf, model_fn, optimizer_fn, lr, epochs, notes=""):
    print(f"\n--- Running {exp_id} ---")
    
    # Подготовка DataLoader
    train_ds = SubsetTransform(train_subset, train_tf)
    val_ds = SubsetTransform(val_subset, val_tf)
    # Test тоже через SubsetTransform для применения трансформов
    test_ds = SubsetTransform(
        Subset(test_dataset, list(range(len(test_dataset)))), 
        val_tf  # На test используем val трансформ (без аугментаций)
    )
    
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)
    
    # Модель
    model = model_fn().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optimizer_fn(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    
    best_val_acc = 0
    best_model_wts = None
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    for epoch in range(epochs):
        t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        v_loss, v_acc = evaluate(model, val_loader, criterion, DEVICE)
        scheduler.step()
        
        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)
        history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc)
        
        if v_acc > best_val_acc:
            best_val_acc = v_acc
            best_model_wts = deepcopy(model.state_dict())
            
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Train Acc: {t_acc:.2f} | Val Acc: {v_acc:.2f}")
            
    model.load_state_dict(best_model_wts)
    _, test_acc = evaluate(model, test_loader, criterion, DEVICE)
    
    return model, history, best_val_acc, test_acc, exp_id

In [ ]:
# C1 exp
model_c1, hist_c1, val_c1, test_c1, _ = run_experiment(
    "C1", transform_base, transform_base, 
    lambda: SimpleCNN(), optim.SGD, lr=0.01, epochs=15, notes="No Augmentation"
)
log_run("C1", "classification", "CIFAR100", "SimpleCNN", "SGD", 0.01, 15, 
        val_c1, test_c1, notes="Base")


--- Running C1 ---
Epoch 5/15 | Train Acc: 9.70 | Val Acc: 12.75
Epoch 10/15 | Train Acc: 17.41 | Val Acc: 21.64
Epoch 15/15 | Train Acc: 22.00 | Val Acc: 24.74


In [25]:
# C2 exp
model_c2, hist_c2, val_c2, test_c2, _ = run_experiment(
    "C2", transform_aug, transform_base, 
    lambda: SimpleCNN(), optim.SGD, lr=0.01, epochs=15, notes="With Augmentation"
)
log_run("C2", "classification", "CIFAR100", "SimpleCNN", "SGD", 0.01, 15, 
        val_c2, test_c2, notes="Augmented")


--- Running C2 ---
Epoch 5/15 | Train Acc: 6.76 | Val Acc: 9.88
Epoch 10/15 | Train Acc: 12.66 | Val Acc: 16.00
Epoch 15/15 | Train Acc: 16.11 | Val Acc: 19.66


In [28]:
# C3 exp
model_c3, hist_c3, val_c3, test_c3, _ = run_experiment(
    "C3", transform_resnet, transform_resnet, 
    lambda: get_resnet18(freeze_backbone=True), optim.Adam, lr=0.001, epochs=10, 
    notes="Pretrained Frozen"
)
log_run("C3", "classification", "CIFAR100", "ResNet18 (Frozen)", "Adam", 0.001, 10, 
        val_c3, test_c3, notes="Head Only")


--- Running C3 ---
Epoch 5/10 | Train Acc: 20.51 | Val Acc: 20.81
Epoch 10/10 | Train Acc: 21.57 | Val Acc: 21.55


In [29]:
# C4 exp
model_c4, hist_c4, val_c4, test_c4, _ = run_experiment(
    "C4", transform_resnet, transform_resnet, 
    lambda: get_resnet18(freeze_backbone=False), optim.Adam, lr=0.0005, epochs=10, 
    notes="Pretrained Fine-tune"
)
log_run("C4", "classification", "CIFAR100", "ResNet18 (Fine-tune)", "Adam", 0.0005, 10, 
        val_c4, test_c4, notes="Fine-tune")


--- Running C4 ---
Epoch 5/10 | Train Acc: 51.29 | Val Acc: 49.02
Epoch 10/10 | Train Acc: 60.34 | Val Acc: 52.32


In [30]:
# Собираем все эксперименты
experiments = [
    ("C1", val_c1, test_c1, model_c1, hist_c1),
    ("C2", val_c2, test_c2, model_c2, hist_c2),
    ("C3", val_c3, test_c3, model_c3, hist_c3),
    ("C4", val_c4, test_c4, model_c4, hist_c4)
]

# Находим лучший по val_accuracy
best_exp = max(experiments, key=lambda x: x[1])
best_name, best_val, best_test, best_model, best_hist = best_exp

print(f"Best Experiment: {best_name} with Val Acc: {best_val:.2f}%")

# Сохраняем лучшую модель
torch.save(best_model.state_dict(), "artifacts/best_classifier.pt")
print("Saved artifacts/best_classifier.pt")

# Сохраняем конфиг
config = {
    "experiment_id": best_name,
    "dataset": "CIFAR100",
    "model": "SimpleCNN" if best_name in ["C1", "C2"] else "ResNet18",
    "transforms": "Augmented" if best_name in ["C2", "C3", "C4"] else "Base",
    "optimizer": "SGD" if best_name in ["C1", "C2"] else "Adam",
    "seed": SEED
}
with open("artifacts/best_classifier_config.json", "w") as f:
    json.dump(config, f, indent=2)
print("Saved artifacts/best_classifier_config.json")

Best Experiment: C4 with Val Acc: 52.32%
Saved artifacts/best_classifier.pt
Saved artifacts/best_classifier_config.json


In [31]:
# График кривых обучения лучшей модели
plt.figure(figsize=(10, 5))
plt.plot(best_hist['train_acc'], label='Train Acc')
plt.plot(best_hist['val_acc'], label='Val Acc')
plt.title(f"Best Model ({best_name}) Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy %")
plt.legend()
plt.grid(True)
plt.savefig("artifacts/figures/classification_curves_best.png", dpi=150)
plt.close()
print("Saved artifacts/figures/classification_curves_best.png")

# График сравнения C1-C4
names = [x[0] for x in experiments]
vals = [x[1] for x in experiments]
tests = [x[2] for x in experiments]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, vals, width, label='Val Accuracy')
rects2 = ax.bar(x + width/2, tests, width, label='Test Accuracy')

ax.set_ylabel('Accuracy %')
ax.set_title('Comparison C1-C4')
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.legend()
plt.savefig("artifacts/figures/classification_compare.png", dpi=150)
plt.close()
print("Saved artifacts/figures/classification_compare.png")

Saved artifacts/figures/classification_curves_best.png
Saved artifacts/figures/classification_compare.png


In [32]:
fig, axes = plt.subplots(2, 4, figsize=(15, 5))
orig_img, _ = train_dataset_full[10]
axes[0, 0].imshow(orig_img)
axes[0, 0].set_title("Original")

for i, ax in enumerate(axes.flat[1:]):
    aug_img = transform_aug(orig_img)
    ax.imshow(aug_img.permute(1, 2, 0))
    ax.set_title(f"Aug {i+1}")
    ax.axis('off')

plt.tight_layout()
plt.savefig("artifacts/figures/augmentations_preview.png", dpi=150)
plt.close()
print("Saved artifacts/figures/augmentations_preview.png")

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.8957008..2.0246198].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.8957008..2.0246198].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.8062342..2.0246198].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.8957008..1.9962232].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.8957008..2.0246198].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.8957008..1.9962232].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.895700

Saved artifacts/figures/augmentations_preview.png


In [58]:
pet_dataset = torchvision.datasets.OxfordIIITPet(
    root="./data_pets", 
    split="trainval", 
    target_types=["segmentation"], 
    download=True,
    transform=None
)
print(f"Loaded {len(pet_dataset)} pet images")



# Берём подмножество для демонстрации
pet_subset_indices = list(range(0, 50, 5))  # 10 семплов
pet_subset = Subset(pet_dataset, pet_subset_indices)

# Важно: batch_size=1 и без num_workers, чтобы избежать коллации PIL
pet_loader = DataLoader(pet_subset, batch_size=1, shuffle=False, num_workers=0)

Loaded 3680 pet images


In [51]:

# Загружаем предобученную DeepLabV3
seg_model = deeplabv3_resnet50(
    weights=torchvision.models.segmentation.DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1
)
seg_model = seg_model.to(DEVICE)
seg_model.eval()
print("Loaded DeepLabV3_ResNet50")

Loaded DeepLabV3_ResNet50


In [47]:
def calculate_iou(pred, gt):
    intersection = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    if union == 0:
        return 0
    return intersection / union

In [61]:

iou_v1_list = []
iou_v2_list = []

# Берём 10 примеров для визуализации
sample_indices = list(range(0, min(50, len(pet_dataset)), 5))
n_samples = len(sample_indices)

fig, axes = plt.subplots(n_samples, 4, figsize=(15, 4*n_samples))
if n_samples == 1:
    axes = axes.reshape(1, -1)

for idx, sample_idx in enumerate(sample_indices):
    # 🔥 ВАЖНО: OxfordIIITPet с target_types=["seg"] возвращает (img, mask), а не (img, {"mask": ...})
    img_pil, gt_mask_pil = pet_dataset[sample_idx]  # оба — PIL Image
    
    # Конвертируем в нужные форматы
    img_tensor = transforms.ToTensor()(img_pil).unsqueeze(0).to(DEVICE)  # [1, 3, H, W]
    gt_mask_np = np.array(gt_mask_pil)  # [H, W], numpy uint8
    
    # Инференс
    with torch.no_grad():
        output = seg_model(img_tensor)['out']  # [1, 21, H, W] — COCO классы
    
    # V1: Argmax + фильтрация по классам cat(16) и dog(18) из COCO
    pred_v1 = output.argmax(1).squeeze().cpu().numpy()  # [H, W]
    pred_binary_v1 = np.isin(pred_v1, [16, 18]).astype(np.uint8)
    
    # V2: Морфологическая очистка
    kernel = np.ones((5, 5), np.uint8)
    opening = cv2.morphologyEx(pred_binary_v1, cv2.MORPH_OPEN, kernel)
    closing = cv2.morphologyEx(opening, cv2.MORPH_CLOSE, kernel)
    pred_binary_v2 = closing
    
    # Бинаризуем GT: питомец = любой класс > 0
    gt_binary = (gt_mask_np > 0).astype(np.uint8)
    
    # Считаем IoU
    iou_v1 = calculate_iou(pred_binary_v1, gt_binary)
    iou_v2 = calculate_iou(pred_binary_v2, gt_binary)
    
    iou_v1_list.append(iou_v1)
    iou_v2_list.append(iou_v2)
    
    # Визуализация
    ax_row = axes[idx] if n_samples > 1 else axes
    ax_row[0].imshow(img_pil)
    ax_row[0].set_title("Input")
    ax_row[0].axis('off')
    ax_row[1].imshow(gt_binary, cmap='gray')
    ax_row[1].set_title(f"GT Mask")
    ax_row[1].axis('off')
    ax_row[2].imshow(pred_binary_v1, cmap='gray')
    ax_row[2].set_title(f"V1 (IoU={iou_v1:.2f})")
    ax_row[2].axis('off')
    ax_row[3].imshow(pred_binary_v2, cmap='gray')
    ax_row[3].set_title(f"V2 (IoU={iou_v2:.2f})")
    ax_row[3].axis('off')

plt.tight_layout()
plt.savefig("artifacts/figures/segmentation_examples.png", dpi=150, bbox_inches='tight')
plt.close()
print("Saved artifacts/figures/segmentation_examples.png")

# Считаем средние метрики и логируем
mean_iou_v1 = np.mean(iou_v1_list) if iou_v1_list else 0
mean_iou_v2 = np.mean(iou_v2_list) if iou_v2_list else 0

print(f"Mean IoU V1: {mean_iou_v1:.4f}")
print(f"Mean IoU V2: {mean_iou_v2:.4f}")

log_run("V1", "segmentation", "OxfordIIITPet", "DeepLabV3 ResNet50", 
        "-", "-", 0, 0, 0, mean_iou_v1, notes="Argmax + Class Filter")
log_run("V2", "segmentation", "OxfordIIITPet", "DeepLabV3 ResNet50", 
        "-", "-", 0, 0, 0, mean_iou_v2, notes="Morphological Cleanup")

# График метрик
plt.figure(figsize=(8, 6))
plt.bar(['V1 (Base)', 'V2 (Morph)'], [mean_iou_v1, mean_iou_v2], 
        color=['skyblue', 'salmon'])
plt.ylabel('Mean IoU')
plt.title('Segmentation Performance Comparison')
plt.ylim(0, 1)
for i, v in enumerate([mean_iou_v1, mean_iou_v2]):
    plt.text(i, v + 0.02, f"{v:.3f}", ha='center')
plt.savefig("artifacts/figures/segmentation_metrics.png", dpi=150, bbox_inches='tight')
plt.close()
print("Saved artifacts/figures/segmentation_metrics.png")

Saved artifacts/figures/segmentation_examples.png
Mean IoU V1: 0.0829
Mean IoU V2: 0.0829
Saved artifacts/figures/segmentation_metrics.png


In [63]:

mean_iou_v1 = np.mean(iou_v1_list)
mean_iou_v2 = np.mean(iou_v2_list)

print(f"Mean IoU V1: {mean_iou_v1:.4f}")
print(f"Mean IoU V2: {mean_iou_v2:.4f}")

log_run("V1", "segmentation", "OxfordIIITPet", "DeepLabV3 ResNet50", 
        "-", "-", 0, 0, 0, mean_iou_v1, notes="Argmax + Class Filter")
log_run("V2", "segmentation", "OxfordIIITPet", "DeepLabV3 ResNet50", 
        "-", "-", 0, 0, 0, mean_iou_v2, notes="Morphological Cleanup")

plt.figure(figsize=(8, 6))
plt.bar(['V1 (Base)', 'V2 (Morph)'], [mean_iou_v1, mean_iou_v2], 
        color=['skyblue', 'salmon'])
plt.ylabel('Mean IoU')
plt.title('Segmentation Performance Comparison')
plt.ylim(0, 1)
for i, v in enumerate([mean_iou_v1, mean_iou_v2]):
        plt.text(i, v + 0.02, f"{v:.3f}", ha='center')
plt.savefig("artifacts/figures/segmentation_metrics.png", dpi=150)
plt.close()
print("Saved artifacts/figures/segmentation_metrics.png")

Mean IoU V1: 0.0829
Mean IoU V2: 0.0829
Saved artifacts/figures/segmentation_metrics.png


In [64]:
save_runs()

Saved artifacts/runs.csv
